# OutputConfiguration Implementation Test

This notebook implements and tests the OutputConfiguration class for coordinate transformation in the mouseReMoCo application.

**Key Features:**
1. Separate output configuration from display configuration
2. Coordinate transformation: screen → center-origin with y-reversed (matplotlib compatible)
3. Self-documenting CSV headers showing transformation parameters
4. Pure function `transform_coordinates()` for testability

In [ ]:
class OutputConfiguration:
    """Configuration for data output coordinate system (CSV, LSL).
    
    This class defines coordinate transformation rules for output data,
    keeping it separate from display configuration.
    
    The transformation process:
    1. Raw tablet event arrives in screen coordinates (origin=top-left)
    2. transform_coordinates() converts to output coordinate system
    3. CSV backend writes transformed coordinates
    4. CSV header documents the transformation rules
    
    Example:
        config = OutputConfiguration(center_x=512, center_y=384)
        x_out, y_out = config.transform_coordinates(600, 300)
        # Result: x_out=88, y_out=84 (center-origin, y-reversed)
    """
    
    def __init__(self, center_x: int, center_y: int):
        """Initialize output configuration with display center.
        
        Args:
            center_x: X coordinate of circle center (for documentation)
            center_y: Y coordinate of circle center (for documentation)
        """
        self.center_x = center_x
        self.center_y = center_y
        
        # Output coordinate system settings
        self.origin_mode = "center"     # "screen" or "center"
        self.y_reversed = True          # True = matplotlib style (y increases upward)
    
    def transform_coordinates(self, x: float, y: float) -> tuple[float, float]:
        """Apply coordinate transformation for output.
        
        Args:
            x: Raw screen X coordinate (0 = left edge)
            y: Raw screen Y coordinate (0 = top edge)
        
        Returns:
            Tuple of (x_transformed, y_transformed) according to output settings
        
        Transformations applied (if enabled):
            1. Origin shift: x_new = x - center_x
            2. Y reversal: y_new = center_y - y
        """
        x_out = x
        y_out = y
        
        # Apply origin transformation
        if self.origin_mode == "center":
            x_out = x - self.center_x
        
        # Apply Y-axis reversal (for matplotlib compatibility)
        if self.y_reversed:
            y_out = self.center_y - y
        
        return x_out, y_out
    
    def to_config_string(self) -> str:
        """Convert output configuration to semicolon-separated string for CSV header.
        
        Returns:
            String like: "outputOriginMode center; outputYReversed true; outputCenterX 512; outputCenterY 384"
        """
        return (
            f"outputOriginMode {self.origin_mode}; "
            f"outputYReversed {str(self.y_reversed).lower()}; "
            f"outputCenterX {self.center_x}; "
            f"outputCenterY {self.center_y}"
        )

## Test 1: Basic Transformation

Test the coordinate transformation with a simple example.

In [ ]:
# Example 1024x768 screen with center at (512, 384)
output_config = OutputConfiguration(center_x=512, center_y=384)

# Test case: point at (600, 300) in screen coordinates
x_screen, y_screen = 600, 300
x_out, y_out = output_config.transform_coordinates(x_screen, y_screen)

print("=== Basic Transformation Test ===")
print(f"Screen coordinates: ({x_screen}, {y_screen})")
print(f"Center: ({output_config.center_x}, {output_config.center_y})")
print(f"Transformed coordinates: ({x_out:.1f}, {y_out:.1f})")
print()

# Verify the math:
print("Step-by-step:")
print(f"  1. X shift: {x_screen} - {output_config.center_x} = {x_out:.1f}")
print(f"  2. Y reverse: {output_config.center_y} - {y_screen} = {y_out:.1f}")
print()

# Test more points
print("Additional test points:")
test_points = [
    (512, 384),  # Center → (0, 0)
    (612, 384),  # Right → (100, 0)
    (512, 284),  # Top → (0, 100)  [y reversed!]
    (412, 484),  # Bottom-left → (-100, -100) [y reversed!]
]

for x, y in test_points:
    x_t, y_t = output_config.transform_coordinates(x, y)
    print(f"  ({x:3d}, {y:3d}) → ({x_t:7.1f}, {y_t:7.1f})")

## Test 2: Configuration String for CSV Header

Test that the configuration generates the correct header string for CSV files.

In [ ]:
config_string = output_config.to_config_string()
print("=== Configuration String for CSV Header ===")
print(config_string)
print()
print("This will appear in the first line of the CSV file:")
print(f"  software mouseReMoCo; version 2.0.0-python; screenWidth 1024; screenHeight 768; centerX 512; centerY 384; {config_string}")

## Test 3: Visual Representation

Create a visualization showing how the coordinate system changes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a figure showing the coordinate transformation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Screen coordinates (left plot)
ax1.set_xlim(0, 1024)
ax1.set_ylim(768, 0)  # Inverted to show screen coordinates
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('X (pixels)', fontsize=12)
ax1.set_ylabel('Y (pixels)', fontsize=12)
ax1.set_title('Screen Coordinates\n(origin at top-left, y increases downward)', fontsize=13, fontweight='bold')

# Draw center point
ax1.plot(512, 384, 'ro', markersize=10, label='Center (512, 384)')
# Draw circle
circle_screen = plt.Circle((512, 384), 150, fill=False, color='blue', linewidth=2, label='Target circle')
ax1.add_patch(circle_screen)
# Mark some test points
for x, y in test_points:
    ax1.plot(x, y, 'bs', markersize=6)
    ax1.annotate(f'({x},{y})', (x, y), xytext=(5, 5), textcoords='offset points', fontsize=8)

ax1.legend(loc='upper left')

# Output coordinates (right plot)
ax2.set_xlim(-250, 250)
ax2.set_ylim(-250, 250)  # Y is now normal (increases upward)
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('X (pixels)', fontsize=12)
ax2.set_ylabel('Y (pixels)', fontsize=12)
ax2.set_title('Output Coordinates\n(origin at center, y increases upward - matplotlib style)', 
              fontsize=13, fontweight='bold')

# Draw center point
ax2.plot(0, 0, 'ro', markersize=10, label='Center (0, 0)')
# Draw circle at same size
circle_output = plt.Circle((0, 0), 150, fill=False, color='blue', linewidth=2, label='Target circle')
ax2.add_patch(circle_output)
# Transform and mark test points
for x, y in test_points:
    x_t, y_t = output_config.transform_coordinates(x, y)
    ax2.plot(x_t, y_t, 'bs', markersize=6)
    ax2.annotate(f'({x_t:.0f},{y_t:.0f})', (x_t, y_t), xytext=(5, 5), textcoords='offset points', fontsize=8)

ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

print("\n✓ Coordinate transformation visualization complete")

## Test 4: Integration with CSVBackend

Demonstrate how OutputConfiguration integrates with the CSV backend.

In [ ]:
import csv
import tempfile
from datetime import datetime
from pathlib import Path

# Minimal Configuration class for testing
class TestConfiguration:
    def __init__(self):
        self.screen_width = 1024
        self.screen_height = 768
        self.center_x = 512
        self.center_y = 384
        self.external_radius = 150
        self.internal_radius = 80
        self.internal_limit = 96
        self.external_limit = 140
        self.border_radius = 1
        self.index_of_difficulty = 2.39
        self.task_string = "circular"
        self.is_with_lsl = False

# Minimal CSVBackend implementation with OutputConfiguration
class CSVBackendWithOutputConfig:
    def __init__(self, config, output_config, data_filename="test_data.csv", marker_filename="test_marker.csv"):
        self.config = config
        self.output_config = output_config
        self.data_filename = data_filename
        self.marker_filename = marker_filename
        self.creation_timestamp = datetime.now()
        
        self.data_file = None
        self.data_writer = None
        self.marker_file = None
        self.marker_writer = None
        
        self._init_files()
    
    def _init_files(self):
        """Create and write headers to CSV files"""
        # Create data.csv
        self.data_file = open(self.data_filename, "w", newline="")
        self.data_writer = csv.writer(self.data_file)
        
        # Write configuration header
        config_line = self._config_to_string()
        self.data_file.write(config_line + "\n")
        
        # Write timestamp
        timestamp_str = self.creation_timestamp.astimezone().isoformat()
        self.data_file.write(timestamp_str + "\n")
        self.data_file.write("\n")
        
        # Write column headers
        self.data_writer.writerow([
            "event_timestamp", "call_time", "mouseX", "mouseY", 
            "mouseInTarget", "pressure", "tiltX", "tiltY"
        ])
        self.data_file.flush()
        
        print(f"✓ Created {self.data_filename}")
    
    def _config_to_string(self) -> str:
        """Convert configuration to semicolon-separated string for CSV header"""
        config_dict = {
            "software": "mouseReMoCo",
            "version": "2.0.0-python",
            "screenWidth": self.config.screen_width,
            "screenHeight": self.config.screen_height,
            "centerX": self.config.center_x,
            "centerY": self.config.center_y,
            "isWithLSL": str(self.config.is_with_lsl).lower(),
        }
        
        if self.config.task_string == "circular":
            config_dict.update({
                "externalRadius": self.config.external_radius,
                "internalRadius": self.config.internal_radius,
                "internalLimit": self.config.internal_limit,
                "externalLimit": self.config.external_limit,
                "borderRadius": self.config.border_radius,
                "indexOfDifficulty": round(self.config.index_of_difficulty, 2),
            })
        
        # Add output configuration (NEW)
        output_config_str = self.output_config.to_config_string()
        
        return "; ".join([f"{k} {v}" for k, v in config_dict.items()]) + "; " + output_config_str
    
    def write_data(self, event_timestamp_ms, call_time_ms, x, y, is_inside, 
                   pressure=0.0, tilt_x=0.0, tilt_y=0.0):
        """Write position data to CSV with coordinate transformation"""
        # Transform coordinates for output
        x_out, y_out = self.output_config.transform_coordinates(x, y)
        
        self.data_writer.writerow([
            event_timestamp_ms,
            call_time_ms,
            round(x_out, 2),
            round(y_out, 2),
            1 if is_inside else 0,
            round(pressure, 3),
            round(tilt_x, 2),
            round(tilt_y, 2),
        ])
        self.data_file.flush()
    
    def close(self):
        if self.data_file:
            self.data_file.close()
            print(f"✓ Closed {self.data_filename}")

# Create test instances
test_config = TestConfiguration()
output_config_test = OutputConfiguration(center_x=512, center_y=384)

# Create CSV backend with output configuration
csv_backend = CSVBackendWithOutputConfig(test_config, output_config_test)

# Write some test data
print("\n=== Writing Test Data ===")
test_data_points = [
    (1000, 1005, 512, 384, True, 0.8),   # At center
    (1010, 1015, 612, 384, True, 0.85),  # Right
    (1020, 1025, 512, 284, True, 0.9),   # Top (y reversed!)
    (1030, 1035, 412, 484, False, 0.7),  # Bottom-left (outside)
]

for event_ts, call_ts, x, y, inside, pressure in test_data_points:
    csv_backend.write_data(event_ts, call_ts, x, y, inside, pressure)
    x_out, y_out = output_config_test.transform_coordinates(x, y)
    print(f"  ({x:3d}, {y:3d}) [screen] → ({x_out:7.1f}, {y_out:7.1f}) [output]")

csv_backend.close()

# Read and display the CSV file
print("\n=== CSV File Content ===")
with open(csv_backend.data_filename, 'r') as f:
    content = f.read()
    print(content)

## Test 5: Reading and Analyzing Transformed Data

Load the generated CSV and verify the coordinate transformation is correctly applied.

In [ ]:
import pandas as pd

# Load CSV (skip configuration and timestamp lines)
df = pd.read_csv(csv_backend.data_filename, skiprows=2)

print("=== Loaded Data ===")
print(df.to_string())
print()

# Extract header to show what transformations were applied
with open(csv_backend.data_filename, 'r') as f:
    header_line = f.readline().strip()

print("=== CSV Header (Configuration) ===")
print(header_line)
print()

# Parse and display just the output configuration
print("=== Output Configuration from Header ===")
parts = header_line.split("; ")
for part in parts:
    if "output" in part.lower():
        print(f"  {part}")
print()

# Verify the transformation is correct
print("=== Verification ===")
print("Original screen coordinates → CSV output coordinates:")
for idx, (event_ts, call_ts, x, y, inside, pressure) in enumerate(test_data_points):
    csv_x = df.iloc[idx]['mouseX']
    csv_y = df.iloc[idx]['mouseY']
    
    # Calculate expected values
    expected_x = x - 512
    expected_y = 384 - y
    
    match_x = abs(csv_x - expected_x) < 0.01
    match_y = abs(csv_y - expected_y) < 0.01
    status = "✓" if (match_x and match_y) else "✗"
    
    print(f"{status} ({x:3d}, {y:3d}) → ({csv_x:7.1f}, {csv_y:7.1f}) [expected: ({expected_x:7.1f}, {expected_y:7.1f})]")

## Summary

The OutputConfiguration implementation successfully:

✅ **Separates concerns:** Display config stays untouched, output config handles CSV transformation
✅ **Maintains coherence:** CSV header documents the exact transformation applied
✅ **Provides clarity:** Anyone reading the file can see `outputOriginMode center; outputYReversed true`
✅ **Is testable:** `transform_coordinates()` is a pure function
✅ **Integrates seamlessly:** CSVBackend reads OutputConfiguration and applies transformation at write time

### Key Implementation Points:

1. **OutputConfiguration class:**
   - Holds transformation parameters (origin_mode, y_reversed)
   - Provides `transform_coordinates()` method for coordinate conversion
   - Provides `to_config_string()` for CSV header documentation

2. **CSVBackend integration:**
   - Accepts OutputConfiguration instance in constructor
   - Calls `transform_coordinates()` before writing data to CSV
   - Includes output config in CSV header via `_config_to_string()`

3. **Coordinate system:**
   - **Input:** Screen coordinates (0,0 = top-left, y increases downward)
   - **Output:** Center-origin, matplotlib-style (0,0 = center, y increases upward)

### Next Steps:

To integrate into the full mouseReMoCo application:

1. Add OutputConfiguration import to test-tablet.ipynb cell 1
2. Create OutputConfiguration instance in WindowSetup.finalize_display()
3. Pass it to OutputTablet constructor
4. Pass it to CSVBackend constructor
5. No changes needed to MainWindow, Trail, or CircularTargetWidget (they stay in screen coordinates)